In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
import os
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# ==============================
# CHECK INPUT DATA
# ==============================

print("Available Kaggle inputs:")
print(os.listdir("/kaggle/input"))

# ==============================
# AUTO FIND IMAGE DATASET FOLDER
# ==============================

image_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

DATASET_PATH = None

for root, dirs, files in os.walk("/kaggle/input"):
    class_folder_count = 0

    for d in dirs:
        folder_path = os.path.join(root, d)

        try:
            image_count = sum(
                file.lower().endswith(image_extensions)
                for file in os.listdir(folder_path)
            )

            if image_count > 0:
                class_folder_count += 1

        except:
            pass

    if class_folder_count >= 2:
        DATASET_PATH = root
        break

if DATASET_PATH is None:
    raise FileNotFoundError(
        "No image class folders found. Make sure you added the Human Action Recognition image dataset."
    )

print("Dataset found at:", DATASET_PATH)
print("Class folders:", os.listdir(DATASET_PATH)[:10])

# ==============================
# PARAMETERS
# ==============================

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 10

# ==============================
# DATA GENERATORS
# ==============================

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

train_generator = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training",
    shuffle=True
)

valid_generator = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation",
    shuffle=False
)

# ==============================
# CNN MODEL
# ==============================

model = Sequential([
    Input(shape=(224, 224, 3)),

    Conv2D(32, (3, 3), activation="relu"),
    BatchNormalization(),
    MaxPooling2D(2, 2),

    Conv2D(64, (3, 3), activation="relu"),
    BatchNormalization(),
    MaxPooling2D(2, 2),

    Conv2D(128, (3, 3), activation="relu"),
    BatchNormalization(),
    MaxPooling2D(2, 2),

    Flatten(),

    Dense(256, activation="relu"),
    Dropout(0.5),

    Dense(train_generator.num_classes, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

# ==============================
# CALLBACKS
# ==============================

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    "best_human_action_model.keras",
    monitor="val_accuracy",
    save_best_only=True,
    mode="max"
)

# ==============================
# TRAIN MODEL
# ==============================

history = model.fit(
    train_generator,
    validation_data=valid_generator,
    epochs=EPOCHS,
    callbacks=[early_stop, checkpoint]
)

# ==============================
# EVALUATION
# ==============================

val_loss, val_accuracy = model.evaluate(valid_generator)

print("Validation Loss:", val_loss)
print("Validation Accuracy:", val_accuracy)

# ==============================
# ACCURACY GRAPH
# ==============================

plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")
plt.legend()
plt.show()

# ==============================
# LOSS GRAPH
# ==============================

plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.show()

# ==============================
# SAVE MODEL
# ==============================

model.save("human_action_recognition_model.keras")

print("Model saved successfully.")
print("Classes:", list(train_generator.class_indices.keys()))

2026-05-31 19:00:51.321055: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780254051.616547      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780254051.701073      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780254052.364393      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780254052.364480      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780254052.364484      58 computation_placer.cc:177] computation placer alr

Available Kaggle inputs:
['notebooks']


FileNotFoundError: No image class folders found. Make sure you added the Human Action Recognition image dataset.

In [3]:
import os

print(os.listdir("/kaggle/input"))

['datasets', 'notebooks']


In [4]:
import os

for root, dirs, files in os.walk("/kaggle/input/datasets"):
    print(root)

/kaggle/input/datasets
/kaggle/input/datasets/meetnagadia
/kaggle/input/datasets/meetnagadia/human-action-recognition-har-dataset
/kaggle/input/datasets/meetnagadia/human-action-recognition-har-dataset/Human Action Recognition
/kaggle/input/datasets/meetnagadia/human-action-recognition-har-dataset/Human Action Recognition/test
/kaggle/input/datasets/meetnagadia/human-action-recognition-har-dataset/Human Action Recognition/train


In [1]:
import os

for root, dirs, files in os.walk("/kaggle/input/datasets"):
    print(root)

/kaggle/input/datasets
/kaggle/input/datasets/meetnagadia
/kaggle/input/datasets/meetnagadia/human-action-recognition-har-dataset
/kaggle/input/datasets/meetnagadia/human-action-recognition-har-dataset/Human Action Recognition
/kaggle/input/datasets/meetnagadia/human-action-recognition-har-dataset/Human Action Recognition/test
/kaggle/input/datasets/meetnagadia/human-action-recognition-har-dataset/Human Action Recognition/train


In [2]:
import os
print(os.listdir("/kaggle/input"))

['datasets']


In [3]:
import os

for root, dirs, files in os.walk("/kaggle/input/datasets"):
    print(root)

/kaggle/input/datasets
/kaggle/input/datasets/meetnagadia
/kaggle/input/datasets/meetnagadia/human-action-recognition-har-dataset
/kaggle/input/datasets/meetnagadia/human-action-recognition-har-dataset/Human Action Recognition
/kaggle/input/datasets/meetnagadia/human-action-recognition-har-dataset/Human Action Recognition/test
/kaggle/input/datasets/meetnagadia/human-action-recognition-har-dataset/Human Action Recognition/train


In [4]:
import os
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout

DATASET_PATH = "/kaggle/input/datasets/meetnagadia/human-action-recognition-har-dataset/Human Action Recognition"

TRAIN_DIR = os.path.join(DATASET_PATH, "train")
TEST_DIR = os.path.join(DATASET_PATH, "test")

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 10

train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training"
)

valid_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation"
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

model = Sequential([
    Input(shape=(224, 224, 3)),

    Conv2D(32, (3, 3), activation="relu"),
    MaxPooling2D(2, 2),

    Conv2D(64, (3, 3), activation="relu"),
    MaxPooling2D(2, 2),

    Conv2D(128, (3, 3), activation="relu"),
    MaxPooling2D(2, 2),

    Flatten(),

    Dense(256, activation="relu"),
    Dropout(0.5),

    Dense(train_generator.num_classes, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

history = model.fit(
    train_generator,
    validation_data=valid_generator,
    epochs=EPOCHS
)

loss, accuracy = model.evaluate(test_generator)

print("Test Accuracy:", accuracy)

plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.legend()
plt.title("Accuracy")
plt.show()

plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.legend()
plt.title("Loss")
plt.show()

model.save("human_action_recognition_model.keras")
print("Model saved successfully")

2026-05-31 20:50:17.008108: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780260617.276656      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780260617.346680      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780260617.903068      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780260617.903122      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780260617.903125      58 computation_placer.cc:177] computation placer alr

Found 0 images belonging to 0 classes.
Found 0 images belonging to 0 classes.
Found 0 images belonging to 0 classes.


2026-05-31 20:50:49.418925: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


ValueError: Received an invalid value for `units`, expected a positive integer. Received: units=0

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout

# =========================
# PATHS
# =========================

BASE_PATH = "/kaggle/input/datasets/meetnagadia/human-action-recognition-har-dataset/Human Action Recognition"

TRAIN_DIR = os.path.join(BASE_PATH, "train")
TEST_DIR = os.path.join(BASE_PATH, "test")

print("Train files:", os.listdir(TRAIN_DIR)[:5])
print("Test files:", os.listdir(TEST_DIR)[:5])

# =========================
# LOAD TRAINING CSV
# =========================

CSV_PATH = os.path.join(BASE_PATH, "Training_set.csv")

df = pd.read_csv(CSV_PATH)

print(df.head())
print(df.columns)

image_col = "filename"
label_col = "label"

# =========================
# ADD FULL IMAGE PATH
# =========================

df[image_col] = df[image_col].astype(str)

df["filepath"] = df[image_col].apply(
    lambda x: os.path.join(TRAIN_DIR, x)
)

df = df[df["filepath"].apply(os.path.exists)]

print("Usable images:", len(df))
print(df.head())

# =========================
# TRAIN VALID SPLIT
# =========================

train_df, valid_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df[label_col]
)

# =========================
# DATA GENERATORS
# =========================

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 10

datagen = ImageDataGenerator(rescale=1./255)

train_generator = datagen.flow_from_dataframe(
    train_df,
    x_col="filepath",
    y_col=label_col,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

valid_generator = datagen.flow_from_dataframe(
    valid_df,
    x_col="filepath",
    y_col=label_col,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

NUM_CLASSES = len(train_generator.class_indices)

print("Number of classes:", NUM_CLASSES)
print("Classes:", train_generator.class_indices)

# =========================
# CNN MODEL
# =========================

model = Sequential([
    Input(shape=(224, 224, 3)),

    Conv2D(32, (3, 3), activation="relu"),
    MaxPooling2D(2, 2),

    Conv2D(64, (3, 3), activation="relu"),
    MaxPooling2D(2, 2),

    Conv2D(128, (3, 3), activation="relu"),
    MaxPooling2D(2, 2),

    Flatten(),

    Dense(256, activation="relu"),
    Dropout(0.5),

    Dense(NUM_CLASSES, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

# =========================
# TRAIN
# =========================

history = model.fit(
    train_generator,
    validation_data=valid_generator,
    epochs=EPOCHS
)

# =========================
# EVALUATE
# =========================

loss, accuracy = model.evaluate(valid_generator)

print("Validation Accuracy:", accuracy)

# =========================
# PLOTS
# =========================

plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.legend()
plt.title("Accuracy")
plt.show()

plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.legend()
plt.title("Loss")
plt.show()

# =========================
# SAVE MODEL
# =========================

model.save("human_action_recognition_model.keras")

print("Model saved successfully")

Train files: ['Image_4378.jpg', 'Image_5576.jpg', 'Image_6267.jpg', 'Image_747.jpg', 'Image_8010.jpg']
Test files: ['Image_4378.jpg', 'Image_747.jpg', 'Image_561.jpg', 'Image_345.jpg', 'Image_3019.jpg']
      filename         label
0  Image_1.jpg       sitting
1  Image_2.jpg  using_laptop
2  Image_3.jpg       hugging
3  Image_4.jpg      sleeping
4  Image_5.jpg  using_laptop
Index(['filename', 'label'], dtype='object')
Usable images: 12600
      filename         label  \
0  Image_1.jpg       sitting   
1  Image_2.jpg  using_laptop   
2  Image_3.jpg       hugging   
3  Image_4.jpg      sleeping   
4  Image_5.jpg  using_laptop   

                                            filepath  
0  /kaggle/input/datasets/meetnagadia/human-actio...  
1  /kaggle/input/datasets/meetnagadia/human-actio...  
2  /kaggle/input/datasets/meetnagadia/human-actio...  
3  /kaggle/input/datasets/meetnagadia/human-actio...  
4  /kaggle/input/datasets/meetnagadia/human-actio...  
Found 10080 validated image filena

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_6 (Conv2D)               │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │    22,151,424 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 15)             │         3,855 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,248,527 (84.87 MB)

 Trainable params: 22,248,527 (84.87 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
315/315 ━━━━━━━━━━━━━━━━━━━━ 646s 2s/step - accuracy: 0.1350 - loss: 2.6412 - val_accuracy: 0.1952 - val_loss: 2.5155
Epoch 2/10
315/315 ━━━━━━━━━━━━━━━━━━━━ 673s 2s/step - accuracy: 0.2360 - loss: 2.3732 - val_accuracy: 0.2603 - val_loss: 2.3264
Epoch 3/10
 42/315 ━━━━━━━━━━━━━━━━━━━━ 8:30 2s/step - accuracy: 0.3771 - loss: 2.0156